# Milestone 3 Exploration Notebook - Step 2

This notebook contains supporting exploration and experiments for Milestone 3 - Step 2: Additional Feature (Option 3).

The full written discussion and final conclusions are provided in:
`results/milestone3_discussion.md`

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

## Load Data

In [2]:
# Define File Paths
RAW_DIR = Path("../data/raw")
OUT_DIR = Path("../data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

main_file = RAW_DIR / "Patio_Lawn_and_Garden.jsonl"
meta_file = RAW_DIR / "meta_Patio_Lawn_and_Garden.jsonl"

## Step 2: Add One Additional Feature - Option 3: Scale to $\ge$ 100k Products

To improve retrieval quality, we scaled our sample dataset to include at least 100,000 products. To prepare the dataset for retrieval, we removed the individual rating column and retained only the average rating. This allows us to aggregate reviews at the product level without introducing duplicate information. We also reduced the dataset size by optimizing data types for several columns, including the number of ratings, average rating, price, and category. 

### Kept Columns

#### From Reviews Data
- `title` renamed to `review_title` as a short textual summary of the review that can support retrieval
- `text` as the main review content and a key source of natural language signal for both BM25 and semantic search
- `parent_asin` used to join reviews with product metadata

#### From Metadata Data
- `title` renamed to `product_title` as an essential and high signal field for both retrieval and display
- `description` as detailed product text that supports semantic search
- `features` as keyword rich product attributes useful for retrieval
- `categories` as additional keywords that can improve BM25 matching
- `average_rating` kept as metadata for display
- `rating_number` provides context for the average rating and improves interpretation
- `price` retained as optional metadata for potential future use such as price based queries
- `parent_asin` used to join metadata with reviews data

### Dropped Columns

The following columns were excluded because they are not useful for text retrieval, are highly sparse, or add unnecessary complexity.

#### Reviews Data
- `images` non text and often empty
- `asin` identifier that is redundant with `parent_asin`
- `user_id` identifier with no retrieval value
- `rating` not meaningful after aggregating multiple reviews into a single product-level record
- `timestamp` not relevant for retrieval or required display
- `helpful_vote` not used in the current retrieval or display setup
- `verified_purchase` optional metadata excluded to keep the dataset lean

#### Metadata Data
- `main_category` too broad and low signal
- `images` and `videos` non text or complex nested fields
- `store` low informational value
- `details` contains structured product attributes such as brand, color, and dimensions. While potentially useful, it requires additional preprocessing and overlaps with information already present in `features` and `description`, so it was excluded for simplicity in this milestone.
- `bought_together` entirely missing in the sample
- `subtitle` and `author` extremely sparse

## Data Preprocessing

### Text Preprocessing Decisions

- Renamed columns (`title` → `review_title`, `product_title`) for clarity
- Removed non-text or irrelevant columns (e.g., images, user_id, timestamp)
- Converted list-based fields (`features`, `description`, `categories`) into plain text using string concatenation
- Merged review and metadata tables using `parent_asin`

In [3]:
# Load the review data
reviews_df = pd.read_json(main_file, lines=True)

In [4]:
# Load the meta data
meta_df = pd.read_json(meta_file, lines=True)

In [5]:
# Drop columns
reviews_df.drop(
    columns=['images', 'asin', 'user_id', 'rating',
             'timestamp', 'helpful_vote', 'verified_purchase'
             ],
    inplace=True
)
meta_df.drop(
    columns=['main_category', 'images', 
             'videos', 'store', 'details',
             'bought_together', 'subtitle', 
             'author'
             ],
    inplace=True
)

In [6]:
# Rename columns
reviews_df.rename(
    columns={
        "title": "review_title"
    },
    inplace=True
)

meta_df.rename(
    columns={
        "title": "product_title"
    },
    inplace=True
)

In [7]:
# Convert columns with lists of text to simple strings rather than lists 
def flatten_to_text(x):
    if isinstance(x, list):
        return " ".join(str(item) for item in x if pd.notna(item))
    if pd.isna(x):
        return ""
    return str(x)

for col in ["features", "description", "categories"]:
    meta_df[col] = meta_df[col].apply(flatten_to_text)

# Check Result to ensure it worked properly
meta_df[["features", "description", "categories"]].head()

,features,description,categories
0,Fits for Echo PB-2155 Leaf Debris Blower ES-21...,,"Patio, Lawn & Garden Outdoor Power Tools Repla..."
1,WELCOME TO B-HYVE: Looking to get a smart spri...,From the manufacturer Orbit is North America’s...,"Patio, Lawn & Garden Gardening & Lawn Care Wat..."
2,"Mini garden flag (size: 12.5"" x 18"") are made ...","KafePross have a lot of flag design, there are...","Patio, Lawn & Garden Outdoor Décor Flags"
3,Kwik-Covers are hygienic table covers and disp...,"8' x 30"" Kwik-Cover is a lightweight, elastic ...","Patio, Lawn & Garden Patio Furniture & Accesso..."
4,SUITABLE SIZE Perfect for Greenhouse size 90''...,Agfabric Plant cover made with high quality st...,"Patio, Lawn & Garden Gardening & Lawn Care Pla..."


In [8]:
# Merge the review data and metadata using "parent_asin"
merged_df = pd.merge(
    reviews_df, 
    meta_df, 
    on='parent_asin', 
    how='inner'
)

merged_df.head()

,review_title,text,parent_asin,product_title,average_rating,rating_number,features,description,price,categories
0,Someone’s return? Arrived broken in several pl...,Seriously. Not my best month with Amazon. Open...,B07CL2CD4D,Perky-Pet 1.5 lb Red Metal Tube Wild Bird Feed...,4.4,217,Powder-coated finish ensures rust resistance R...,Bring charm and simplicity to your outdoor spa...,18.0,"Patio, Lawn & Garden Outdoor Décor Backyard Bi..."
1,Arrived broken,The chair wouldn't fold out properly. The foot...,B0C6HJPJXL,Ostrich Lounge Chaise,4.2,5841,Polyester Imported Note : To ensure proper ass...,Patented open face cavity with horseshoe cushi...,67.95,"Patio, Lawn & Garden Patio Furniture & Accesso..."
2,Disappointing,Bought this on a Cyber Monday deal and I'm so ...,B01CHG9MXA,Quality Source Products Best 50',3.1,228,,,None,"Patio, Lawn & Garden Gardening & Lawn Care Wat..."
3,Cute style but just ok- don't buy more than 1 ...,Bought these because of the unique look. I pur...,B004BK1DL0,"Panacea 89357 Circular Obelisk with Finial, Bl...",4.6,36,"Circular obelisk with finial Thin, round metal...",The panacea circular obelisk with finial bring...,69.99,"Patio, Lawn & Garden Gardening & Lawn Care Pla..."
4,Seriously I have tried just about everything. ...,Seriously I have tried just about everything. ...,B004CSWPS6,"Bonide MOLEMAX Mole & Vole Repellent Granules,...",3.7,2994,MOLEMAX Mole & Vole Repellent Granules keep an...,Protect your lawn & garden from moles & voles ...,19.52,"Patio, Lawn & Garden Pest Control Repellents S..."


## Sample from merged_df, Aggregate Reviews, Optimize Data Types, and Save to Parquet File

In Step 1, we validated and estimated the number of unique products using product_title, since parent_asin was not available in the original sample dataset. 
In Step 2, we are resampling from the original review and metadata data, so we use parent_asin, the unique identifier, instead to avoid issues caused by missing or incomplete product titles. Accordingly, we sample 225,000 rows from merged_df, ensuring that both parent_asin and product_title contain more than 100,000 unique products.

In [9]:
# Sample rows from the fully merged dataframe 
scaled_sample = merged_df.sample(n=225000, random_state=42)

# Checking whether the scaled sample has more than 100k products
scaled_sample.nunique()

review_title      142083
text              211492
parent_asin       101268
product_title     100647
average_rating        41
rating_number       5561
features           86453
description        52345
price               9570
categories           756
dtype: int64

In [10]:
# Aggregate reviews at the product level using parent_asin
scaled_sample = (
    scaled_sample
    .groupby("parent_asin", as_index=False)
    .agg({
        "review_title": "first",
        "product_title": "first",
        "average_rating": "first",
        "rating_number": "first",
        "features": "first",
        "description": "first",
        "price": "first",
        "categories": "first",
        "text": list
    })
)

# Combine list of review texts into a single string per product
scaled_sample["review_text"] = scaled_sample["text"].apply(
    lambda x: " ".join([str(i) for i in x if isinstance(i, str)])
)

# Drop raw text and parent_asin 
scaled_sample = scaled_sample.drop(columns=['text', 'parent_asin'])

scaled_sample.head()

,review_title,product_title,average_rating,rating_number,features,description,price,categories,review_text
0,Broke on the 3rd use,Multi Alphabet Folding Umbrella 43'' Dia,1.0,1,,"Shield yourself from drizzly, dreary weather i...",None,"Patio, Lawn & Garden Patio Furniture & Accesso...",I bought this because I have had other Jonatha...
1,Just what I have been looking for!,A Georgia Food Forest: 180 Perennial Edible Pl...,4.2,17,This is an incredible book by artist and perma...,,None,"Patio, Lawn & Garden Gardening & Lawn Care Pla...",I want to start a food forrest in central Geor...
2,"Really fun, enthusiastic book",Edible Landscaping with a Permaculture Twist: ...,4.5,271,Edible Landscaping at its Best! Edible Landsca...,About the Author Michael Judd has worked with ...,24.95,"Patio, Lawn & Garden Outdoor Power Tools Metal...","Really fun, enthusiastic book. Basic, yes, but..."
3,Outstanding!!!!!!!,"Audubon Birdhouse Book, Revised and Updated: B...",4.6,251,The Audubon Birdhouse Book is the most authori...,"About the Author Margaret A. Barker , a Chesap...",15.89,"Patio, Lawn & Garden Outdoor Décor Backyard Bi...",This book is all inclusive. Not just bird hous...
4,LASAGNA GARDENING,Lasagna Gardening: A New Layering System for B...,4.7,550,A gardening system that works-- so you don't h...,"Review ""Pat Lanza is a genius! It's a pleasure...",17.62,"Patio, Lawn & Garden Farm & Ranch Agricultural...","THIS BOOK WAS SUPER, i HAD TROUBLE LAYING IT D..."


In [11]:
# verify we still have at least 100,000 unique products after aggregation
scaled_sample["product_title"].nunique()

100647

In [12]:
# Check dataset structure and data types
scaled_sample.info()

<class 'pandas.DataFrame'>
RangeIndex: 101268 entries, 0 to 101267
Data columns (total 9 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   review_title    101268 non-null  str    
 1   product_title   101268 non-null  str    
 2   average_rating  101268 non-null  float64
 3   rating_number   101268 non-null  int64  
 4   features        101268 non-null  str    
 5   description     101268 non-null  str    
 6   price           60079 non-null   object 
 7   categories      101268 non-null  str    
 8   review_text     101268 non-null  str    
dtypes: float64(1), int64(1), object(1), str(6)
memory usage: 169.2+ MB


In [13]:
# Optimize data types to reduce memory usage and improve processing efficiency
scaled_sample["rating_number"] = pd.to_numeric(scaled_sample["rating_number"], downcast="integer")
scaled_sample["average_rating"] = pd.to_numeric(scaled_sample["average_rating"], downcast="float")
scaled_sample["categories"] = scaled_sample["categories"].astype("category")

# Clean and extract numeric values from price column
scaled_sample["price"] = (
    scaled_sample["price"]
    .replace(["—", "–", "", "N/A", "na", "null"], np.nan)
    .astype(str)
    .str.extract(r"(\d+\.?\d*)")[0]
)

In [14]:
# Validate dataset structure and confirm data type conversions
scaled_sample.info()

<class 'pandas.DataFrame'>
RangeIndex: 101268 entries, 0 to 101267
Data columns (total 9 columns):
 #   Column          Non-Null Count   Dtype   
---  ------          --------------   -----   
 0   review_title    101268 non-null  str     
 1   product_title   101268 non-null  str     
 2   average_rating  101268 non-null  float32 
 3   rating_number   101268 non-null  int32   
 4   features        101268 non-null  str     
 5   description     101268 non-null  str     
 6   price           60079 non-null   str     
 7   categories      101268 non-null  category
 8   review_text     101268 non-null  str     
dtypes: category(1), float32(1), int32(1), str(6)
memory usage: 160.1 MB


In [15]:
# Save scaled sample dataset as a compressed Parquet file
output_path = OUT_DIR / "processed_scaled_sample.parquet"
scaled_sample.to_parquet(output_path, index=False, compression="snappy")

In [16]:
# Final validation step: ensure parquet file is correctly saved and readable
scaled_df = pd.read_parquet(OUT_DIR / "processed_scaled_sample.parquet")
scaled_df.head()

,review_title,product_title,average_rating,rating_number,features,description,price,categories,review_text
0,Broke on the 3rd use,Multi Alphabet Folding Umbrella 43'' Dia,1.0,1,,"Shield yourself from drizzly, dreary weather i...",NaN,"Patio, Lawn & Garden Patio Furniture & Accesso...",I bought this because I have had other Jonatha...
1,Just what I have been looking for!,A Georgia Food Forest: 180 Perennial Edible Pl...,4.2,17,This is an incredible book by artist and perma...,,NaN,"Patio, Lawn & Garden Gardening & Lawn Care Pla...",I want to start a food forrest in central Geor...
2,"Really fun, enthusiastic book",Edible Landscaping with a Permaculture Twist: ...,4.5,271,Edible Landscaping at its Best! Edible Landsca...,About the Author Michael Judd has worked with ...,24.95,"Patio, Lawn & Garden Outdoor Power Tools Metal...","Really fun, enthusiastic book. Basic, yes, but..."
3,Outstanding!!!!!!!,"Audubon Birdhouse Book, Revised and Updated: B...",4.6,251,The Audubon Birdhouse Book is the most authori...,"About the Author Margaret A. Barker , a Chesap...",15.89,"Patio, Lawn & Garden Outdoor Décor Backyard Bi...",This book is all inclusive. Not just bird hous...
4,LASAGNA GARDENING,Lasagna Gardening: A New Layering System for B...,4.7,550,A gardening system that works-- so you don't h...,"Review ""Pat Lanza is a genius! It's a pleasure...",17.62,"Patio, Lawn & Garden Farm & Ranch Agricultural...","THIS BOOK WAS SUPER, i HAD TROUBLE LAYING IT D..."
